In [1]:
from haystack.document_stores.in_memory import InMemoryDocumentStore

document_store = InMemoryDocumentStore()

In [5]:
import json 

with open("/Users/katharinazeilnhofer/VS Code/ir-project/data_retrieval/kochwiki_data/rezepte_parsed.json", "r") as f:
    recipes = json.load(f)

with open("/Users/katharinazeilnhofer/VS Code/ir-project/data_retrieval/kochwiki_data/zutaten_parsed.json", "r") as f:
    ingredients = json.load(f)

data = recipes + ingredients

In [56]:
from haystack import Document

docs = []
for item in recipes:
    if not item.get("zutaten_namen") or not item.get("zubereitung_raw"):
        continue
    
    content = f"Rezept: {item['title']}\n"
    if item.get('metadata', {}).get('zeit'):
        content += f"Zeit: {item['metadata']['zeit']}\n"
    if item.get('metadata', {}).get('schwierigkeit'):
        content += f"Schwierigkeit: {item['metadata']['schwierigkeit']}\n"
    
    content += "Zutaten:\n"
    for z in item.get("zutaten", []):
        menge = z.get("menge", "")
        einheit = z.get("einheit", "")
        name = z.get("name", "")
        content += f"- {menge} {einheit} {name}\n"
    
    content += f"Zubereitung: {item['zubereitung_raw']}\n"
    
    docs.append(Document(content=content, meta={"title": item["title"]}))

print(f"{len(docs)} Dokumente")


8125 Dokumente


In [57]:
from haystack.components.embedders import SentenceTransformersDocumentEmbedder

doc_embedder = SentenceTransformersDocumentEmbedder(model="T-Systems-onsite/cross-en-de-roberta-sentence-transformer")
doc_embedder.warm_up()

In [58]:
document_store = InMemoryDocumentStore()

docs_with_embeddings = doc_embedder.run(docs)
document_store.write_documents(docs_with_embeddings["documents"])

Batches:   0%|          | 0/254 [00:00<?, ?it/s]

8125

In [59]:
from haystack.components.embedders import SentenceTransformersTextEmbedder

text_embedder = SentenceTransformersTextEmbedder(model="T-Systems-onsite/cross-en-de-roberta-sentence-transformer")

In [60]:
from haystack.components.retrievers.in_memory import InMemoryEmbeddingRetriever

retriever = InMemoryEmbeddingRetriever(document_store)

In [61]:
from haystack.components.builders import ChatPromptBuilder
from haystack.dataclasses import ChatMessage

template = [
    ChatMessage.from_user(
        """Du bist ein hilfreicher Kochassistent. Beantworte die Frage basierend auf den folgenden Rezepten.
Zeige das relevanteste Rezept mit allen Details (Zutaten, Zubereitung).
Wenn mehrere Rezepte passen, zeige das beste und erwähne kurz die Alternativen.

Rezepte:
{% for document in documents %}
---
{{ document.content }}
---
{% endfor %}

Frage: {{question}}
Antwort:"""
    )
]

prompt_builder = ChatPromptBuilder(template=template)

ChatPromptBuilder has 2 prompt variables, but `required_variables` is not set. By default, all prompt variables are treated as optional, which may lead to unintended behavior in multi-branch pipelines. To avoid unexpected execution, ensure that variables intended to be required are explicitly set in `required_variables`.


In [66]:
from dotenv import load_dotenv
load_dotenv()
import os

from haystack.components.generators.chat import HuggingFaceAPIChatGenerator

generator = HuggingFaceAPIChatGenerator(
    api_type="serverless_inference_api",
    api_params={"model": "meta-llama/Meta-Llama-3-8B-Instruct"}
)
generator.warm_up()
print("Generator ready!")

Generator ready!


In [63]:
from haystack import Pipeline

text_embedder = SentenceTransformersTextEmbedder(model="T-Systems-onsite/cross-en-de-roberta-sentence-transformer")
retriever = InMemoryEmbeddingRetriever(document_store, top_k=3)
prompt_builder = ChatPromptBuilder(template=template)

basic_rag_pipeline = Pipeline()
basic_rag_pipeline.add_component("text_embedder", text_embedder)
basic_rag_pipeline.add_component("retriever", retriever)
basic_rag_pipeline.add_component("prompt_builder", prompt_builder)
basic_rag_pipeline.add_component("llm", generator)


ChatPromptBuilder has 2 prompt variables, but `required_variables` is not set. By default, all prompt variables are treated as optional, which may lead to unintended behavior in multi-branch pipelines. To avoid unexpected execution, ensure that variables intended to be required are explicitly set in `required_variables`.


In [64]:
basic_rag_pipeline.connect("text_embedder.embedding", "retriever.query_embedding")
basic_rag_pipeline.connect("retriever", "prompt_builder")
basic_rag_pipeline.connect("prompt_builder.prompt", "llm.messages")

🚅 Components
  - text_embedder: SentenceTransformersTextEmbedder
  - retriever: InMemoryEmbeddingRetriever
  - prompt_builder: ChatPromptBuilder
  - llm: HuggingFaceAPIChatGenerator
🛤️ Connections
  - text_embedder.embedding -> retriever.query_embedding (list[float])
  - retriever.documents -> prompt_builder.documents (list[Document])
  - prompt_builder.prompt -> llm.messages (list[ChatMessage])

In [65]:
question = "Hast du ein schnelles Mittagessen mit Nudeln?"

response = basic_rag_pipeline.run({"text_embedder": {"text": question}, "prompt_builder": {"question": question}})

print(response["llm"]["replies"][0].text)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Ja, ich habe ein schnelles Mittagessen mit Nudeln. Das Rezept "Nudeln mit Spinat und Lachs" ist perfekt für ein schnelles Mittagessen. Es dauert nur 15 Minuten und ist sehr leicht zu zubereiten.

**Rezept: Nudeln mit Spinat und Lachs**

Zeit: 15 Minuten
Schwierigkeit: leicht

Zutaten:
- 250 g Nudeln (z.B Spaghetti, Bandnudeln, etc.)
- 400 g Rahmspinat (tiefgekühlt)
-   Salz
- 1 EL Öl
- 200 g Räucherlachs

Zubereitung:
• Die Nudeln nach Anweisung Kochen.
• Den gefrorenen Rahmspinat in einen Topf geben und bei mittlerer Hitze abgedeckt aufkochen lassen und ca. 2 Minuten weiter köcheln lassen.
• Den Räucherlachs in beliebig große Streifen schneiden.
• Nudeln, Spinat und den Lachs auf Teller verteilen.

Dieses Rezept ist sehr einfach und schnell zuzubereiten und bietet eine gute Balance aus Kohlenhydraten, Protein und Vitamen. Es ist auch eine gute Wahl für alle, die gerne Lachs essen.
